# AdaptiveSRE Colab Validation Notebook

This notebook is set up for Google Colab validation.

Flow:
1. Install dependencies
2. Clone the repo and start mock services as Python processes
3. Start the main server and verify `/health`
4. Run a short training smoke test
5. Run full training
6. Evaluate the trained checkpoint
7. Generate the reward plot

Success criteria:
- `train.py` completes 10 episodes without crashing
- `positive_trajectories > 0`
- `train.py` completes 200 episodes
- `eval.py` shows Gen1 > Gen0
- `rewards_curve.png` is generated

If Colab fails, the likely culprits are:
- `CUDA out of memory`: reduce `batch_size` in `train.py`
- `unsloth` import error: re-run the install cell
- Mock services do not start: check port conflicts
- `Connection refused` to `localhost:8000`: server did not start
- All rewards are `0.001`: debug `grader.py`
- Model outputs invalid JSON: tune temperature or prompt formatting in `train.py`

In [ ]:
# Cell 1: Create isolated Colab env + install exact AdaptiveSRE deps
# Colab sometimes fails with python -m venv (ensurepip issue), so use virtualenv.
!python3 -m pip install -q --upgrade pip virtualenv setuptools wheel
!python3 -m virtualenv /content/adaptive_venv

# Install AdaptiveSRE stack into isolated env only.
!/content/adaptive_venv/bin/python -m pip install -q --upgrade pip setuptools wheel
!/content/adaptive_venv/bin/pip install -q unsloth trl==0.9.6 transformers==4.44.0 torch==2.4.0 httpx==0.27.0 matplotlib accelerate peft datasets bitsandbytes
!/content/adaptive_venv/bin/pip install -q fastapi==0.115.0 uvicorn==0.30.0 pydantic==2.7.0 gradio==4.44.0 "protobuf>=4.25.8,<6"

# Remove wandb from both envs to avoid known trl+wandb proto clashes.
!/content/adaptive_venv/bin/pip uninstall -y wandb || true
!pip uninstall -y wandb || true

# Verify interpreter path used for all later commands.
!/content/adaptive_venv/bin/python -c "import sys; print('Using Python:', sys.executable)"

In [ ]:
# Cell 2: Clone & start services from isolated env (Colab has no Docker)
!if [ ! -d Adaptive-SRE ]; then git clone https://github.com/ashifsekh/Adaptive-SRE.git; fi
%cd Adaptive-SRE

# Confirm the interpreter and run import sanity check from isolated env.
!/content/adaptive_venv/bin/python -c "import sys; print('Interpreter in Cell 2:', sys.executable)"
!/content/adaptive_venv/bin/python test_install.py

# Start all 5 mock services in background and write logs for debugging.
!nohup /content/adaptive_venv/bin/python -m uvicorn mock_services.db.main:app --host 0.0.0.0 --port 15432 > db.log 2>&1 &
!nohup /content/adaptive_venv/bin/python -m uvicorn mock_services.auth.main:app --host 0.0.0.0 --port 8102 > auth.log 2>&1 &
!nohup /content/adaptive_venv/bin/python -m uvicorn mock_services.payment.main:app --host 0.0.0.0 --port 8101 > payment.log 2>&1 &
!nohup /content/adaptive_venv/bin/python -m uvicorn mock_services.cache.main:app --host 0.0.0.0 --port 6379 > cache.log 2>&1 &
!nohup /content/adaptive_venv/bin/python -m uvicorn mock_services.notification.main:app --host 0.0.0.0 --port 8103 > notification.log 2>&1 &

# Start main server.
!nohup /content/adaptive_venv/bin/python -m uvicorn server.app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

# Wait for /health with retries; show server log tail on failure.
import time
import urllib.request

ok = False
for _ in range(40):
    try:
        with urllib.request.urlopen('http://localhost:8000/health', timeout=2) as response:
            print(response.read().decode())
        ok = True
        break
    except Exception:
        time.sleep(1)

if not ok:
    print('Server did not become healthy. Last server.log lines:')
    try:
        with open('server.log', 'r') as f:
            print(''.join(f.readlines()[-50:]))
    except Exception as exc:
        print(f'Could not read server.log: {exc}')
    raise SystemExit(1)

In [ ]:
# Cell 3: Short training smoke test (10 episodes, 5-10 mins on T4)
!/content/adaptive_venv/bin/python train.py --episodes 10 --task easy --output ./test_checkpoints/ --env_url http://localhost:8000

In [ ]:
# Cell 4: If step 3 works, run real training (200 episodes, ~1-2 hours)
!/content/adaptive_venv/bin/python train.py --episodes 200 --task hard --output ./checkpoints/gen1/ --env_url http://localhost:8000

In [ ]:
# Cell 5: Evaluate
!/content/adaptive_venv/bin/python eval.py --trained_model ./checkpoints/gen1/final --env_url http://localhost:8000 --output eval_results.json

# Cell 6: Plot
!/content/adaptive_venv/bin/python plot_rewards.py
from IPython.display import Image
Image('rewards_curve.png')